# Instalação

Rodar um dos códigos abaixo no terminal

## Instalação simples

Pode gerar conflito com outros pacotes

In [ ]:
pip install pyleida

```
#macOS users:
pip install pyleida

#Linux users:
pip3 install pyleida
```

## Instalação com ambiente virtual

Qualquer uma dessas opções já é suficiente, sendo a última já direcionada para este projeto.

### Ambiente virtual conda

```
conda create --name pL python=3.9
conda activate pL

#Linux users: 
pip3 install pyleida

#macOS users:
pip install pyleida
```

### Ambiente venv

venv é um módulo do python para a criação de ambientes virtuais.

Ambiente virtual: é uma pasta que tem a instalação do python e tudo que está junto com o python (incluindo pacotes de terceiros).

A utilização de um ambiente para cada projeto resolve os seguintes problemas:
* versões diferentes de pacotes que foram usados em diferentes momentos
* dependências de pacotes
* instalações de muitos pacotes enchendo uma máquina
* versões diferentes do python

Podem ser dados quais quer nomes para os ambientes virtuais, mas os mais comuns são:

venv, env

.venv, .env (estes criam uma pasta oculta em sistemas Unix - mac e linux)

https://docs.python.org/3/library/venv.html

```
#macOS users:
python -m venv pL
source pL/bin/activate
pip install pyleida

#Linux users:
python3 -m venv pL
source pL/bin/activate
pip3 install pyleida
```


### Ambiente estável para o pyleida

Necessita do arquivo environment.yml disponível no github, salvo na mesma pasta desse script

```
conda env create -f environment.yml
conda activate pL
```

# Iniciando a análise

## Formato dos dados

![image.png](attachment:image.png)

O pacote Leida considera que a pasta deste projeto contém uma outra pasta (com nome a ser especificado na função) com os dados de fMRI em um certo formato.

Vamos considerar uma pasta chamada "data".

Esta pasta deve conter:

* __pasta "time_series"__
    * _uma pasta para cada sujeito_
        * um arquivo tseries.csv contendo a série temporal de BOLD (N_ROIs x N_volumes)  
Alternatively, instead of providing a single .csv file per participant the user can simply provide a single pickle file called 'time_series.pkl' that contains a dictionary with the functional MRI data of each subject. In this case, keys must be the 'subject ids' and the values 2D arrays (N_ROIs x N_volumes) with the time series of each brain region/parcel.


* __arquivo "metadata.csv"__  
    id dos sujeitos e grupo/condição a dividir os participantes
    |subject_id | condition |
    |-----------|-----------|
    |sub-0001	| HC |
    |sub-0002	| HC |
    |sub-0003	| HC |
    |sub-0004	| AUT |
    |sub-0005	| AUT |
    |sub-0006	| AUT |  
    
    However, in a more complex scenario, instead of assigning a simple group/condition label to the complete scan, we may want to assign a particular condition to each particular volume (for e.g., if we have conducted an experiment with a block design). In this case, the 'metadata.csv' file must contain, for each subject, a row for each volume. Alternatively, the user can simply provide a 'metadata.pkl' file with a dictionary having the subject_ids as keys, and lists with the labels as values.

* __arquivo "rois_labels.txt"__  
    Arquivo com o nome de cada região do atlas utilizado.  
    A ordem dos labels deve concordar com a ordem da série temporal nos arquivos .csv do BOLD dos sujeitos

* __arquivo "rois_coordinates.csv"__  
    Arquivo com as coordenadas 3D (X,Y,Z) de cada região no espaço MNI.  
    Essas coordenadas são necessárias para criar as figuras representando cada região como um nó/nodo (node)

* __arquivo "{parcellation}.nii{.gz}"__ (opcional)  
    Arquivo .nii ou .nii.gz com a parcelação que deve ser usada para extrair a série temporal de fMRI.  
    Esse arquivo é necessário para criar gráficos mostrando os estados BOLD phase-locking identificados na superfície cortical

## Pipeline

In [ ]:
# importando a classe Leida
from pyleida import Leida
import numpy as np
np.float_ = np.float64

# instanciando a classe Leida, especificando onde estão os dados
ld = Leida('./ArquivosNew/GlasserReligiousUpNoNow')

# Rodando o pipeline completo:
ld.fit_predict(TR=2.2,
               paired_tests=False,
               n_perm=5_000,
               save_results=True)

# TR = Time Repetition dos dados fMRI
# paired_tests = se as condições ou grupos são independentes ou não
# n_perm = número de permutações que serão usadas nas análises estatísticas



In [ ]:
ld.significant_states(metric='occupancies')

In [ ]:
import pandas as pd

data = {}
for s in range(1, 6):  # states 1 to 5
    result = ld.state_rois(k=5, state=s)
    data[f"state_{s}"] = result

# Convert to DataFrame (different lengths -> columns will be padded with NaN)
df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in data.items()]))

# Save the dataframe to CSV
df.to_csv("state_rois.csv", index=False)

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Exemplo: suponha que seu DataFrame se chame df
df = ld.occupancies(k=5)

# Vamos assumir que suas colunas estão assim:
# subject, condition, state1, state2, state3, state4, state5
state_cols = df.columns[2:]  # pega as 5 últimas colunas

# Calcula a média de cada estado por condição
mean_states = df.groupby("condition")[state_cols].mean().T

# Faz o gráfico
mean_states.plot(kind="bar", figsize=(10,6))

plt.title("Média dos estados por condição")
plt.ylabel("Média")
plt.xlabel("Estados")
plt.xticks(rotation=0)
plt.legend(title="Condição")
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Exemplo de criação do DataFrame (substitua pelo seu carregamento real)
df = ld.occupancies(k=5)

# Calcular médias agrupadas por condição
mean_df = df.groupby('condition').mean(numeric_only=True).reset_index()

# Reformatar o DataFrame para visualização
plot_df = mean_df.melt(id_vars='condition', 
                       var_name='State', 
                       value_name='Mean')

# Configurações de estilo
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))

# Criar o gráfico de barras
ax = sns.barplot(data=plot_df, 
                 x='State', 
                 y='Mean', 
                 hue='condition',
                 palette=['skyblue', 'salmon'])

# Personalizações do gráfico
plt.title('Mean difference by State between Religious and Nonreligious conditions')
plt.ylabel('Mean')
plt.xlabel('States')
plt.legend(title='Condition')
ax.set_axisbelow(True)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Mostrar valores nas barras
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', 
                fontsize=8, color='black')

plt.tight_layout()
plt.show()

In [ ]:
ld.stats(k=5,metric='occupancies')

In [ ]:
ld.stats(k=5,metric='dwell_times')

In [ ]:
ld.barplot_centroids(k=5,state='all')

In [ ]:
ld.group_transitions(k=2)
ld.group_transitions(k=3)
ld.group_transitions(k=4)
ld.group_transitions(k=5)
ld.group_transitions(k=6)
ld.group_transitions(k=7)
ld.group_transitions(k=8)
ld.group_transitions(k=9)
ld.group_transitions(k=10)

In [ ]:
ld.plot_states_nodes(k=5,state=1,show_labels=False)

In [ ]:
ld.plot_states_nodes(k=5,state=2,show_labels=False)

In [ ]:
ld.plot_states_nodes(k=5,state=3,show_labels=False)

In [ ]:
ld.plot_states_nodes(k=5,state=4,show_labels=False)

In [ ]:
ld.plot_states_nodes(k=5,state=5,show_labels=False)

In [ ]:
ld.group_static_fc(group="yes")

In [ ]:
ld.group_static_fc(group="no")

In [ ]:
ld.plot_states_matrices(k=2)
ld.plot_states_matrices(k=3)
ld.plot_states_matrices(k=4)
ld.plot_states_matrices(k=5)
ld.plot_states_matrices(k=6)
ld.plot_states_matrices(k=7)
ld.plot_states_matrices(k=8)
ld.plot_states_matrices(k=9)
ld.plot_states_matrices(k=10)

## Output

rodar a função fit_predict do Leida cria um output que salva uma estrutura de pastas e arquivos com os resultados.

A estrutura fica dentro da pasta do projeto, numa pasta chamada __"Leida_results"__  
A pasta contém:

* arquivo __"eigenvectors.csv"__  
    para cada sujeito, o leading eigenvector de cada ponto no tempo/volume  

    |subject_id | condition | eigenvector ROI 1 | eigenvector ROI 2 | eigenvector ROI N |
    |-----------|-----------| ------------------| ------------------| ------------------|
    |sub-0001	|    HC     | 0.123 | 0.123 | 0.123 |
    |sub-0002	|    HC     | 0.362 | 0.362 | 0.362 |
    |sub-0003	|    HC     | 0.872 | 0.872 | 0.872 |
    |sub-0004	|    AUT    | 0.391 | 0.391 | 0.391 |
    |sub-0005	|    AUT    | 0.456 | 0.456 | 0.456 |
    |sub-0006	|    AUT    | 0.789 |   0.789 |   0.789 |  
  
    
* pasta __clustering__
    * pasta __models__
        * arquivo __"model{1-n}.pkl"__  
        um para cada modelo KMeans fittado
  
    * arquivo __"predictions.csv"__
        para cada partição K, o label de cluster predito para cada volume de sujeito/ponto no tempo

    |subject_id | condition | K1 | K2 | KN |
    |-----------|-----------|----| ---| ---|
    |sub-0001	| HC        |  A |  B |  D |
    |sub-0002	| HC        |  B |  C |  F |
    |sub-0003	| HC        |  C |  D |  E |
    |sub-0004	| AUT       |  D |  A |  C |
    |sub-0005	| AUT       |  E |  B |  M |
    |sub-0006	| AUT       |  C |  D |  A |
    
    
    * arquivo __"clustering_performance.csv"__  
        Dunn score, distorção, Silhouette score, Davies-Bouldin score para cada partição K  
    * arquivo de imagem __"clustering_performance.png"__  
        mostra a informação contida no clustering_performance.csv

* pasta __"dynamics_metrics"__  
    * imagens mostrando os resultados da análise estatística comparando cada par de condições através de todas as partições K
    * pasta __"k_{1-N partições}"__
        * arquivos com Fractional Occupancies, Dwell Times, Transitions Probabilities e resultados das análises estatísticas de Dwell Times e Fractional Occupancies

![image.png](attachment:image.png)

# Entendendo os outputs

## Eigenvector

A abordagem mais comum para analisar dinâmica de conectividade funcional é fazer matrizes NxN de correlação entre as ROIs.  
Temos uma matriz para cada sujeito em cada ponto no tempo.  

Como as matrizes são simétricas, geralmente se faz as comparações utilizando o triângulo superior da matriz:

```
. X X X X X X X X X X  
. . X X X X X X X X X  
. . . X X X X X X X X  
. . . . X X X X X X X  
. . . . . X X X X X X  
. . . . . . X X X X X  
. . . . . . . X X X X  
. . . . . . . . X X X  
. . . . . . . . . X X  
. . . . . . . . . . X  
. . . . . . . . . . .
```

Essa matriz também pode sofrer uma __decomposição espectral__.  
Essa decomposição gera duas outras matrizes (combinadas, elas podem ser utilizadas para recompor a matriz original), de acordo com a seguinte fórmula:  
                $C=QΛQˆ{−1}$  

$Λ$ é uma matriz diagonal com os _autovalores_  
$Q$ é uma matriz cujas colunas são os ___autovetores__._

#### O __leading eigenvector__ (ou primeiro autovetor) é o autovetor associado ao maior autovalor.  
Ele captura a direção principal de variabilidade na matriz.


O leading eigenvector V1(t) captura o padrão de conectividade dominante da dinâmica no tempo t.  
Usar o eigenvector em vez do triângulo superior da matriz reduz muito a dimensionalidade e ainda explica a maior parte da variância.


Através do tempo, podemos comparar os leading eigenvectors com similaridade de cossenos ou com correlações de Pearson.

## Clustering

É realizada uma análise de clusters em todos os leading eigenvectors através do tempo e sujeitos. Obtém-se uma quantidade discreta de padrões de conectividade funcional (por ex.: 75 × 98 = 17150 leading eigenvectors).

No método desse paper e do Leida, é utilizado o clustering por __k-means__. o k (n de clusters) vai de 2 a 20, repetindo cada um 20 vezes.  
Assim, obtém-se k centroides de clusters. Cada um deles (k) é um vetor Nx1 representando um padrão recorrente de conectividade funcional.  

Para cada k, avalia-se o cluster usando o __Dunn score__. Para efeito de comparação, também é feita a mesma análise usando um modelo oculto de Markov (HMM). O HMM leva em consideração as dependências temporais dos dados.

## Comparação entre grupos

É realizado um teste-t pareado baseado em permutações. É uma versão robusta e não-paramétrica do teste-t pareado comum.

Você tem dois grupos (ex.: grupo A e grupo B) e quer saber se eles diferem em alguma métrica.

Em vez de assumir que os dados seguem uma distribuição normal (como o teste-t tradicional faz), você:

Mistura os rótulos de grupo aleatoriamente (permutação).

Para cada permutação, calcula o valor do teste-t.

Repete isso 5000 vezes → gera uma distribuição nula baseada nos próprios dados.

Compara o valor observado com essa distribuição nula → calcula o p-valor.